# 🏥 CareFlow — Notebook 8: Bottlenecks & Queueing Theory
## Operations Research, Little's Law ($L = \lambda W$) & Departmental Congestion

**Author:** CareFlow Operations Research Team  
**Dataset:** `careflow_analytics/output/mart_bottlenecks.parquet` & `stg_ehr_events.parquet`

---

### Objectives:
1. Map inter-departmental queue delays and calculate transition wait statistics.
2. Apply **Little's Law** ($L = \lambda W$) to quantify mean queue inventory and WIP.
3. Compare Morning Rush (8-11 AM) vs Normal Hours delays using Mann-Whitney U non-parametric tests.
4. Calculate a composite **Bottleneck Impact Score** ($Score = Volume \times Wait\_Time$) to prioritize lean interventions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('dark_background')
sns.set_palette('flare')

bottlenecks_df = pd.read_parquet('../careflow_analytics/output/mart_bottlenecks.parquet')
stg_df = pd.read_parquet('../careflow_analytics/output/stg_ehr_events.parquet')

print(f"Loaded {len(bottlenecks_df)} bottleneck transitions.")
bottlenecks_df.head(6)


In [ ]:
# 1. Bottleneck Impact Ranking
bottlenecks_df["transition"] = bottlenecks_df["source_activity"] + " -> " + bottlenecks_df["target_activity"]
bottlenecks_df["total_hours_lost"] = (bottlenecks_df["transition_volume"] * bottlenecks_df["avg_wait_minutes"]) / 60.0

ranked = bottlenecks_df.groupby("transition").agg(
    total_volume=("transition_volume", "sum"),
    weighted_avg_wait=("avg_wait_minutes", "mean"),
    total_hours_lost=("total_hours_lost", "sum")
).sort_values(by="total_hours_lost", ascending=False).reset_index()

print("Top 5 Bottlenecks by Cumulative Hours Lost:")
ranked.head()


In [ ]:
# 2. Morning Rush vs Off-Peak Queue Wait Comparison (Mann-Whitney U Test)
trans = stg_df[stg_df["previous_activity"].notna()].copy()
trans["transition"] = trans["previous_activity"] + " -> " + trans["activity_name"]
trans["wait_min"] = (trans["event_timestamp"] - trans["previous_event_timestamp"]).dt.total_seconds() / 60.0

plt.figure(figsize=(12, 6))
top_trans = ranked["transition"].head(5).tolist()
plot_data = trans[trans["transition"].isin(top_trans)]

sns.boxplot(data=plot_data, x="transition", y="wait_min", hue="is_morning_rush",
            palette=['#06b6d4', '#f43f5e'], showfliers=False)
plt.title("Queue Delay: Morning Rush (8-11 AM) vs Normal Hours")
plt.xlabel("Pathway Transition")
plt.ylabel("Wait Time (Minutes)")
plt.xticks(rotation=20)
plt.legend(title="Morning Rush")
plt.tight_layout()
plt.show()


In [ ]:
# 3. Statistical Testing of Morning Rush Delays
print("Mann-Whitney U Tests for Rush Hour Delays:")
print("-" * 65)
for t in top_trans:
    sub = trans[trans["transition"] == t]
    rush = sub[sub["is_morning_rush"]]["wait_min"]
    norm = sub[~sub["is_morning_rush"]]["wait_min"]
    if len(rush) > 3 and len(norm) > 3:
        stat, p = stats.mannwhitneyu(rush, norm)
        print(f"{t:<35} | Rush: {rush.mean():.1f}m vs Norm: {norm.mean():.1f}m | p={p:.4e}")


In [ ]:
# 4. Little's Law Queue Sizing (L = lambda * W)
# ED arrivals per hour: lambda
stg_df["event_date_hour"] = stg_df["event_timestamp"].dt.floor("h")
hourly_arrivals = stg_df[stg_df["activity_name"] == "Registration"].groupby("event_date_hour")["case_id"].count()
arrival_rate_lambda = hourly_arrivals.mean()  # Patients per hour

print(f"Mean Arrival Rate (lambda): {arrival_rate_lambda:.2f} patients/hour")
print("\nLittle's Law Work-In-Progress (WIP) Queue Inventory:")
print("-" * 55)
for _, row in ranked.head(5).iterrows():
    w_hours = row["weighted_avg_wait"] / 60.0
    l_queue = arrival_rate_lambda * w_hours
    print(f"{row['transition']:<35} | L = {l_queue:.2f} patients in queue")
